# 04 · SciPy 루틴 (cupyx.scipy.*)

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

공식 [overview](https://docs.cupy.dev/en/stable/overview.html)의 **SciPy Routines** 를 본격적으로 — `scipy.*` → `cupyx.scipy.*` 치환으로.
FFT·선형대수·이미지 처리·**희소행렬**·희소 선형대수·신호처리를 미니앱과 함께 다룹니다.

## 학습 목표
- `scipy.*` 코드를 `cupyx.scipy.*` 로 포팅하고 CPU와 정확성을 검증한다.
- 희소행렬 포맷·내부구조를 이해하고 2D 푸아송을 직접/반복법으로 푼다.
- 이미지 분석(`ndimage`)·신호 필터(`signal`)·수치 안정 함수(`special`)를 활용한다.

## 목차
1. [FFT `scipy.fft` + DCT 압축](#1)
2. [선형대수 `scipy.linalg`](#2)
3. [이미지 처리 `ndimage` + 연결요소](#3)
4. [희소행렬 `sparse` (+포맷·내부구조)](#4)
5. [희소 선형대수 `sparse.linalg` + 2D 푸아송](#5)
6. [신호처리 `signal`](#6)
7. [special · stats](#7)
8. [체크포인트](#8)

> 백엔드: cuFFT·cuSOLVER·cuSPARSE. 모두 `scipy`와 동일 인터페이스입니다.

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
from course_utils import print_env, bench, cpu_ms, gpu_ms, compare, allclose
print_env()

<a id="1"></a>
## 1. FFT `scipy.fft` + DCT 압축

📖 [`cupyx.scipy.fft`](https://docs.cupy.dev/en/stable/reference/scipy_fft.html) — `fft`, `rfft`, `dct/idct`, `dst`, `next_fast_len`

### 🔬 이론 배경 — DCT
- **DCT**(이산 코사인 변환): 실수 코사인 기저로 변환. 신호 에너지를 **소수의 저주파 계수**에 집중시킴.
  → 상위 계수만 남기면 **손실 압축**(JPEG·MP3의 핵심 원리).
- 우리가 보는 사진이나 듣는 음악 데이터에는 인간의 눈과 귀가 눈치채지 못하는 고주파 잡음이 너무 많음.
- DCT(이산 코사인 변환) 변환을 하면 데이터가 '주파수 세상'으로 변하면서, 가장 중요한 굵직한 정보(저주파 성분)들은 앞쪽에 몰리고, 사람이 들을 수 없는 고주파 성분 들은 뒤쪽으로 밀려남.
  → 손실 압축: 뒤쪽으로 밀려난 쓸모없는 숫자들을 0으로 싹 날려버릴 수 있음. 이것이 바로 화질이나 음질을 살짝 희생하면서 용량을 줄이는 손실 압축의 원리임.

In [ ]:
from scipy.fft import dct as sp_dct, idct as sp_idct
from cupyx.scipy.fft import dct as cp_dct, idct as cp_idct, next_fast_len
x_np = np.random.random(100_000).astype(np.float32); x_cp = cp.asarray(x_np)

# "CPU가 한 압축 결과와 GPU가 한 압축 결과가 똑같은지" 검증하는 코드
# norm='ortho'는 변환 전후의 데이터 총 에너지를 보존하라는 수학적 옵션
allclose(sp_dct(x_np, norm='ortho'), cp_dct(x_cp, norm='ortho'), rtol=1e-3, atol=1e-3, name='DCT')

# next_fast_len(100_000)은 "지금 데이터가 100,000개인데, 연산 속도를 극대화하려면 데이터를 몇 개로 맞춰주는 게 좋아?"라고 물어보는 함수.
print('next_fast_len(100000)=', next_fast_len(100_000))

### 미니앱 — DCT 기반 압축
DCT 계수의 **상위 일부만 유지**하고 역변환하면 손실 압축이 됩니다. 유지 비율에 따른 복원 오차를 보세요.

In [ ]:
def dct_compress(x, keep):
    xp = cp.get_array_module(x)
    dct = cp_dct if xp is cp else sp_dct
    idct = cp_idct if xp is cp else sp_idct
    # keep개만 남기고 나머지는 0으로 날려버리는 압축
    X = dct(x, norm='ortho'); X[keep:] = 0
    return idct(X, norm='ortho')

t = np.linspace(0,1,4096,endpoint=False).astype(np.float32)
sig = (np.sin(2*np.pi*3*t)+0.3*np.sin(2*np.pi*40*t)).astype(np.float32)
for keep in [64, 256, 1024]:
    r = dct_compress(cp.asarray(sig), keep)
    err = float(cp.linalg.norm(r-cp.asarray(sig))/cp.linalg.norm(cp.asarray(sig)))
    print(f'keep={keep:>5}  상대오차={err:.3e}')

In [ ]:
# 추가 - 파형 확인
import numpy as np
import matplotlib.pyplot as plt
from scipy.fft import dct as sp_dct, idct as sp_idct

# 2. 데이터 및 각 신호 준비
t = np.linspace(0, 1, 4096, endpoint=False)
wave_low = np.sin(2 * np.pi * 3 * t)          # ① 완만한 파도
wave_high = 0.3 * np.sin(2 * np.pi * 40 * t)  # ② 자글자글한 잔물결
sig_combined = (wave_low + wave_high).astype(np.float32)  # ③ 둘이 합쳐진 원본

# ④ keep=64 옵션을 거친 압축 복원 신호
sig_compressed = dct_compress(sig_combined, keep=64)

# 3. 4단 세로 그래프 그리기
plt.figure(figsize=(12, 11))

# ① 첫 번째: 완만한 파도 (3Hz)
plt.subplot(4, 1, 1)
plt.plot(t, wave_low, color='blue', linewidth=2)
plt.title("1. Low Frequency Component (wave_low: 3*t) - 'The Essence'")
plt.grid(True)

# ② 두 번째: 자글자글한 잔물결 (40Hz)
plt.subplot(4, 1, 2)
plt.plot(t, wave_high, color='orange', linewidth=1)
plt.title("2. High Frequency Noise (wave_high: 40*t) - 'The Details'")
plt.grid(True)

# ③ 세 번째: 둘이 합쳐진 실제 원본 데이터
plt.subplot(4, 1, 3)
plt.plot(t, sig_combined, color='green', linewidth=1.5)
plt.title("3. Combined Original Signal (sig_combined = 1 + 2) - 'Before Compression'")
plt.grid(True)

# ④ 네 번째: keep=64로 압축 후 복원한 결과
plt.subplot(4, 1, 4)
plt.plot(t, sig_compressed, color='red', linewidth=2)
plt.title("4. Restored Signal after DCT Compression (keep=64) - 'After Compression'")
plt.xlabel("Time (seconds)")
plt.grid(True)

# 레이아웃 깔끔하게 다듬고 화면 출력
plt.tight_layout()
plt.show()

<a id="2"></a>
## 2. 선형대수 `scipy.linalg`

📖 [`cupyx.scipy.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_linalg.html) — `lu_factor/lu_solve`, `solve_triangular`, `expm`, `toeplitz` 등

### 예제 설명
- 1200개의 미지수를 가진 거대한 연립방정식을 푸는 코드
- 우리가 풀어야 할 연립방정식은 Ax = b. 행렬 A라는 문제집이 있고, b라는 정답지가 있을 때, 미지수 x를 구하는 것.
- 컴퓨터는 이 문제를 풀 때 A를 하삼각행렬(L)과 상삼각행렬(U)이라는 풀기 쉬운 두 개의 행렬로 쪼개는데, 이를 LU 분해라고 합니다.
- lu_factor: 복잡한 행렬 A를 연산하기 쉬운 형태(lu, piv)로 쪼개고 변형하는 단계. (전체 연산량의 99%를 차지할 만큼 무겁습니다.)
- lu_solve: 쪼개놓은 뼈대를 가지고 실제 정답(b)을 대입해서 해(x)를 구해내는 단계. (엄청나게 가볍고 빠릅니다.)


In [ ]:
from cupyx.scipy.linalg import lu_factor, lu_solve, expm
n=1200
# n * cp.eye(...)는 대각선 자리에 큰 숫자(1200)를 더해주는 작업. 행렬 A가 **양의 정부호**(positive definite) 행렬이 되도록 만들어서, 연립방정식 Ax=b를 풀 때 안정적으로 풀리도록 보장하는 역할.
A = cp.random.random((n,n),dtype=cp.float32) + n*cp.eye(n,dtype=cp.float32)
b = cp.random.random(n,dtype=cp.float32)
lu,piv = lu_factor(A); x = lu_solve((lu,piv), b)
allclose(A@x, b, rtol=1e-3, atol=1e-3, name='lu_solve')

**연습 — 행렬지수 `expm`**: `scipy.linalg.expm` 코드를 GPU로 포팅하고 CPU와 비교하세요.

In [ ]:
from scipy.linalg import expm as sp_expm
from cupyx.scipy.linalg import expm as cp_expm
M_np = (0.1*np.random.randn(64,64)).astype(np.float32)
# TODO: allclose(..., ..., rtol=1e-2, atol=1e-2, name='expm')

<details><summary>💡 해답 보기</summary>

```python
allclose(sp_expm(M_np), cp.asnumpy(cp_expm(cp.asarray(M_np))), rtol=1e-2, atol=1e-2, name='expm')
```
</details>

<a id="3"></a>
## 3. 이미지 처리 `ndimage` + 연결요소

📖 [`cupyx.scipy.ndimage`](https://docs.cupy.dev/en/stable/reference/scipy_ndimage.html) — `gaussian_filter`, `sobel`, `label`, `center_of_mass`, `rotate`, `zoom`

### 🔬 이론 배경 — 컨볼루션 필터
- 작은 **커널**을 이미지 위로 슬라이딩하며 이웃 픽셀의 **가중합**(컨볼루션).
- **가우시안** = 평활(잡음 제거), **소벨** = 공간 미분(엣지 검출).
- **`label`** = 연결요소(서로 인접한 전경 픽셀 그룹) 식별.

#### 예제 설명 
컴퓨터에게 노이즈(잡음)가 잔뜩 낀 사진을 주면, 물체와 잡음을 구별하지 못해 엉뚱한 개수를 세게 됩니다. 그래서 코드 속 count_blobs 함수는 다음과 같은 3단계를 거칩니다.
1. gaussian_filter (가우시안 필터 - 블러 효과):
사진에 자글자글하게 낀 모래 같은 잡음(노이즈)을 부드럽게 뭉개버리는 단계입니다. 포토샵의 블러(Blur) 효과와 같습니다. 뭉개고 나면 진짜 중요한 굵직한 물체들만 선명하게 남습니다.
2. mask = g > 0.5 (이진화 - 흑백 나누기):
흐릿해진 사진에서 확실하게 밝은 부분(0.5보다 큰 부분)은 흰색(물체)으로, 어두운 부분은 검은색(배경)으로 구분합니다.
3. label (연결요소 식별 - 번호표 붙이기):
흰색 픽셀들이 서로 상하좌우로 붙어있는 것끼리 묶어서 1번 덩어리, 2번 덩어리... 하고 번호표를 붙여줍니다. 섬이 몇 개인지 지도에서 세어주는 것과 같습니다.


In [ ]:
import scipy.ndimage as spnd
import cupyx.scipy.ndimage as cpnd
# 합성 이미지: 밝은 원반 5개 + 잡음
rng=np.random.default_rng(0); H=W=256
img=np.zeros((H,W),np.float32)
centers=[(60,60),(60,190),(190,60),(190,190),(128,128)]
yy,xx=np.mgrid[0:H,0:W]
for cy,cx in centers: img += np.exp(-(((yy-cy)**2+(xx-cx)**2)/200.0)).astype(np.float32)
img += 0.05*rng.standard_normal((H,W)).astype(np.float32)

def count_blobs(image, ndi):
    # TODO: 모듈 선택(ndi) -> gaussian_filter(sigma=2) -> mask(>0.5) -> label() -> return number of labels
    raise NotImplementedError
# print('연결요소 CPU:', count_blobs(img, spnd), '| GPU:', count_blobs(cp.asarray(img), cpnd))

# 시각화를 위해 추가
# ================= [시각화 출력 코드만 아래에 추가] =================
# 그림 출력을 위해 함수 내부에서 일어나는 중간 과정 데이터를 똑같이 가져옵니다.

import matplotlib.pyplot as plt
g_img = spnd.gaussian_filter(img, sigma=2)  # 1. 가우시안 필터 처리된 이미지
mask_img = g_img > 0.5                      # 2. 이진화 마스크 이미지
lab_img, _ = spnd.label(mask_img)           # 3. 라벨링(번호표) 이미지

# 4칸짜리 가로로 긴 모니터를 만들어 단계별 변화를 봅니다.
plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.imshow(img, cmap='gray')
plt.title("1. Original (with Noise)")
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(g_img, cmap='gray')
plt.title("2. Gaussian Filtered")
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(mask_img, cmap='gray')
plt.title("3. Threshold Mask (>0.5)")
plt.axis('off')

plt.subplot(1, 4, 4)
# 'jet' 색상 맵을 쓰면 컴퓨터가 각자 다르게 묶은 덩어리들이 알록달록하게 구별되어 보입니다.
plt.imshow(lab_img, cmap='jet')
plt.title("4. Connected Components")
plt.axis('off')

plt.tight_layout()
plt.show()

<details><summary>💡 해답 보기</summary>

```python
def count_blobs(image, ndi):
    g = ndi.gaussian_filter(image, sigma=2)
    mask = g > 0.5
    lab, num = ndi.label(mask)
    return num

print('연결요소 CPU:', count_blobs(img, spnd), '| GPU:', count_blobs(cp.asarray(img), cpnd))

```
</details>

**연습 — 그래디언트 크기**: 블러 후 `sobel`로 x·y 그래디언트 크기를 구하는 장치 비종속 함수를 완성하세요.

이 코드는 컴퓨터가 사진 속에서 "물체의 테두리(경계선/엣지)를 아주 선명하게 따내는 기술"입니다. 
포토샵의 '선 따기' 필터나 웹툰 스타일의 필터, 혹은 자율주행차가 도로의 차선을 인식할 때 쓰는 핵심 알고리즘입니다.

* 컴퓨터는 어떻게 선을 딸까?
1. gaussian_filter (가우시안 필터):
  선을 따기 전에 자글자글한 잡음이 있으면 잡음까지 전부 선으로 따버리는 참사가 일어납니다. 그래서 먼저 사진을 부드럽게 뭉개서 잡음을 제거합니다.
2. sobel(..., axis=0) (소벨 가로 필터):
  가로 방향(위아래)으로 밝기가 얼마나 급격하게 변하는지 조사합니다. (가로 경계선이 강조됨)
3. sobel(..., axis=1) (소벨 세로 필터):
  세로 방향(좌우)으로 밝기가 얼마나 급격하게 변하는지 조사합니다. (세로 경계선이 강조됨)
4. sqrt(gx*gx + gy*gy) (최종 테두리 합성):
  피타고라스 정리와 비슷합니다. 가로로 변한 양(gx)과 세로로 변한 양(gy)을 합쳐서 물체의 대각선이나 곡선 등 모든 방향의 최종 테두리 두께와 밝기(크기, Magnitude)를 완성합니다.


In [ ]:
def grad_magnitude(image):
    # TODO: 모듈 선택(spnd/cpnd) -> gaussian_filter(sigma=2) -> sobel(axis=0/1) -> sqrt(gx^2+gy^2)
    raise NotImplementedError
# ref=grad_magnitude(img); out=cp.asnumpy(grad_magnitude(cp.asarray(img)))
# allclose(ref,out,rtol=1e-3,atol=1e-3,name='grad_mag')

# 시각화를 위해 추가. 중간 단계 변화를 끄집어내어 그림으로 그립니다.
import matplotlib.pyplot as plt 
g_img = spnd.gaussian_filter(img, sigma=2)
gx_img = spnd.sobel(g_img, axis=0)  # 가로 방향 경계선
gy_img = spnd.sobel(g_img, axis=1)  # 세로 방향 경계선
magnitude = np.sqrt(gx_img*gx_img + gy_img*gy_img)  # 최종 합성 테두리

# 4칸짜리 가로 모니터를 만들어 원리를 봅니다.
plt.figure(figsize=(16, 4))

plt.subplot(1, 4, 1)
plt.imshow(img, cmap='gray')
plt.title("1. Original (with Noise)")
plt.axis('off')

plt.subplot(1, 4, 2)
# 가로 변화량은 위아래 테두리가 흰색/검은색 선으로 두드러집니다.
plt.imshow(np.abs(gx_img), cmap='gray')
plt.title("2. Horizontal Edges (Sobel Y)")
plt.axis('off')

plt.subplot(1, 4, 3)
# 세로 변화량은 좌우 테두리가 흰색/검은색 선으로 두드러집니다.
plt.imshow(np.abs(gy_img), cmap='gray')
plt.title("3. Vertical Edges (Sobel X)")
plt.axis('off')

plt.subplot(1, 4, 4)
# 두 변화량을 합치면 도넛 모양의 완벽한 둥근 테두리 선이 드러납니다.
plt.imshow(magnitude, cmap='gray')
plt.title("4. Final Gradient Magnitude")
plt.axis('off')

plt.tight_layout()
plt.show()


<details><summary>💡 해답 보기</summary>

```python
def grad_magnitude(image):
    if cp.get_array_module(image) is cp:
        ndi = cpnd; xp = cp
    else:
        ndi = spnd; xp = np
    g = ndi.gaussian_filter(image, sigma=2)
    gx = ndi.sobel(g, axis=0); gy = ndi.sobel(g, axis=1)
    return xp.sqrt(gx*gx + gy*gy)

ref = grad_magnitude(img); out = cp.asnumpy(grad_magnitude(cp.asarray(img)))
allclose(ref, out, rtol=1e-3, atol=1e-3, name='grad_mag')
```
</details>

<a id="4"></a>
## 4. 희소행렬 `sparse` (+ 포맷·내부구조)

📖 [`cupyx.scipy.sparse`](https://docs.cupy.dev/en/stable/reference/scipy_sparse.html) (cuSPARSE)

0이 아닌 값만 저장하는 2-D 포맷 네 가지를 제공합니다.

| 포맷 | 클래스 | 특징 |
|------|--------|------|
| 압축 행 | `csr_matrix` | 행 압축 — **SpMV 효율적** |
| 좌표 | `coo_matrix` | `(data,(row,col))` 구성 편리 |
| 압축 열 | `csc_matrix` | 열 압축 |
| 대각 | `dia_matrix` | 대각 위주 |

CSR은 `data`(값), `indices`(열 번호), `indptr`(행 시작 위치) 세 배열로 저장됩니다.

### 🔬 이론 배경 — 희소행렬은 어떻게 저장되나 (CSR)
- **CSR**은 3개 배열로 저장:
  - `data`: 0이 아닌 값들(행 우선 순서)
  - `indices`: 각 값의 **열 번호**
  - `indptr`: 각 **행의 시작 오프셋**(길이 = 행수+1, 마지막 = 총 nnz)
- 행 i의 원소 = `data[indptr[i]:indptr[i+1]]`, 열 = `indices[동일 구간]`.
- 메모리는 **O(nnz)**(비영 개수)뿐 — 밀집 저장 O(n²)보다 훨씬 작고, 행 순회·SpMV에 효율적. (출처: SciPy `csr_matrix`)

In [ ]:
import scipy.sparse as sps
import cupyx.scipy.sparse as cps
# COO -> CSR 내부 구조 관찰
rows=cp.array([0,0,1,2]); cols=cp.array([0,2,1,2]); vals=cp.array([4.,1.,3.,2.],dtype=cp.float32)
A = cps.coo_matrix((vals,(rows,cols)), shape=(3,3)).tocsr()
print('dense=\n', cp.asnumpy(A.toarray()))
print('data   :', cp.asnumpy(A.data))
print('indices:', cp.asnumpy(A.indices))
print('indptr :', cp.asnumpy(A.indptr))

In [ ]:
# 2D 라플라시안을 diags+kron 으로 구성 (SciPy/CuPy 동일 코드)
def laplacian_2d(S, n):
    I = S.identity(n, format='csr', dtype='float32')
    T = S.diags([-1.,2.,-1.], [-1,0,1], shape=(n,n), format='csr', dtype='float32')
    return (S.kron(I,T) + S.kron(T,I)).tocsr()
n=60 # n을 변경해보세요
A_cpu=laplacian_2d(sps,n); A_gpu=laplacian_2d(cps,n)
x_np=np.random.random(n*n).astype(np.float32); x_cp=cp.asarray(x_np)
allclose(A_cpu@x_np, A_gpu@x_cp, rtol=1e-4, atol=1e-4, name='SpMV')
compare('SpMV', lambda:A_cpu@x_np, lambda:A_gpu@x_cp, n_repeat=20, n_warmup=3)

<a id="5"></a>
## 5. 희소 선형대수 `sparse.linalg` + 2D 푸아송

📖 [`cupyx.scipy.sparse.linalg`](https://docs.cupy.dev/en/stable/reference/scipy_sparse_linalg.html) — `cg`, `gmres`, `spsolve`, `eigsh`, `svds`, `norm`

### 🔬 이론 배경 — 직접법 vs 반복법
- **직접법**(`spsolve`, LU): 분해로 정확해를 구하지만, 큰 희소행렬은 **fill-in**으로 메모리 부담.
- **반복법**(`cg` 등): 행렬-벡터 곱(SpMV)만 반복해 근사해에 수렴. 메모리 적음.
- **CG**는 **대칭 양의 정부호(SPD)** 행렬에서 잔차를 최소화하며 수렴 — 라플라시안이 대표 예.

In [ ]:
from cupyx.scipy.sparse.linalg import cg, spsolve, eigsh
b = cp.random.random(n*n, dtype=cp.float32)
# 직접법(spsolve) vs 반복법(cg)
x_cg, info = cg(A_gpu, b, maxiter=2000, rtol=1e-6)
x_dir = spsolve(A_gpu.tocsr(), b)
print('cg info', info, '| cg vs spsolve 상대차:', float(cp.linalg.norm(x_cg-x_dir)/cp.linalg.norm(x_dir)))
vals = eigsh(A_gpu, k=3, which='LM', return_eigenvectors=False)
print('최대 고유값 3개:', cp.asnumpy(cp.sort(vals))[::-1])

**연습 — 1D 라플라시안 + CG**: `diags`로 1D 라플라시안(2,-1)을 만들고 `cg`로 푸세요.

In [ ]:
def laplacian_1d(n):
    # TODO: cps.diags([-1,2,-1],[-1,0,1],shape=(n,n),format='csr',dtype='float32')
    raise NotImplementedError
# n=5000; A=laplacian_1d(n); b=cp.random.random(n,dtype=cp.float32)
# sol,info=cg(A,b,maxiter=5000,rtol=1e-5); print(info, float(cp.linalg.norm(A@sol-b)/cp.linalg.norm(b)))

<details><summary>💡 해답 보기</summary>

```python
def laplacian_1d(n):
    return cps.diags([-1.,2.,-1.],[-1,0,1],shape=(n,n),format='csr',dtype='float32')
```
</details>

<a id="6"></a>
## 6. 신호처리 `signal`

📖 [`cupyx.scipy.signal`](https://docs.cupy.dev/en/stable/reference/scipy_signal.html) — `fftconvolve`, `butter`+`filtfilt`, `spectrogram`, `welch` 등

### 예제 설명
스마트폰 통화를 할 때 주변 소음을 제거하거나, 심전도(ECG) 의료 기기에서 심장 박동 신호만 깨끗하게 걸러낼 때 쓰는 "신호 처리 필터(Butterworth Lowpass Filter)"와 소리를 시각화하는 "스펙트로그램(Spectrogram)"을 다루고 있습니다.

우리가 만든 원본 데이터(sig_np)는 1초에 5번 웅웅거리는 깨끗한 소리(5*t) 위에 자글자글한 화이트 노이즈(0.4*randn)를 강제로 섞어놓은 지저분한 상태입니다.
① butter + filtfilt (저주파 통과 필터)
 - butter(4, 0.05) (버터워스 필터): 필터의 종류입니다. "주파수가 아주 낮은 녀석들만 통과시키고, 기준치보다 높은 시끄러운 고음 노이즈들은 싹 다 커트해버려!" 하는 차단벽을 설계합니다. (4는 벽의 두께/성능, 0.05는 차단할 주파수 기준점입니다.)
 - filtfilt (양방향 필터링): 신호를 앞으로 한 번, 뒤로 한 번 총 두 번 걸러줍니다. 이렇게 앞뒤로 두 번 걸러주면 신호가 밀리거나 왜곡되는 현상(위상 지연) 없이 노이즈만 아주 깔끔하게 지워집니다.

② spectrogram (소리의 시각화 주소록)
 - 오디오 이퀄라이저 화면처럼, 시간의 흐름에 따라 어떤 주파수(고음/저음) 성분이 강하게 들어있는지 지도로 그려내는 작업입니다.
 -결과물로 나온 Sxx.shape는 그 지도의 [가로(시간) x 세로(주파수)] 크기를 나타냅니다.


In [ ]:
from scipy.signal import butter as spb, filtfilt as spf
from cupyx.scipy.signal import butter as cpb, filtfilt as cpf, spectrogram
t=np.linspace(0,1,20000,endpoint=False).astype(np.float32)
sig_np=(np.sin(2*np.pi*5*t)+0.4*np.random.randn(t.size)).astype(np.float32)
def lowpass(xb,xf,sig):
    b,a=xb(4,0.05); return xf(b,a,sig)
allclose(lowpass(spb,spf,sig_np), cp.asnumpy(lowpass(cpb,cpf,cp.asarray(sig_np))), rtol=1e-2, atol=1e-2, name='butter+filtfilt')
# 스펙트로그램(처프)
f,tt,Sxx = spectrogram(cp.asarray(sig_np), fs=20000)
print('spectrogram shape:', Sxx.shape)

# 시각화 추가
import matplotlib.pyplot as plt 
# 중간 결과물 및 스펙트로그램 데이터를 CPU 배열로 가져옵니다.
filtered_sig = lowpass(spb, spf, sig_np)
Sxx_np = cp.asnumpy(Sxx)
f_np = cp.asnumpy(f)
tt_np = cp.asnumpy(tt)

# 시각화를 위한 그림판 준비 (3단 분할)
plt.figure(figsize=(14, 10))

# ① 첫 번째: 자글자글한 잡음이 가득한 원본 신호 파형
plt.subplot(3, 1, 1)
plt.plot(t[:2000], sig_np[:2000], color='orange', alpha=0.7)  # 앞부분 일부만 확대해서 관찰
plt.title("1. Original Signal with Heavy Noise (First 0.1s)")
plt.ylabel("Amplitude")
plt.grid(True)

# ② 두 번째: 필터를 거쳐 잡음이 사르르 녹아내린 깨끗한 파형
plt.subplot(3, 1, 2)
plt.plot(t[:2000], filtered_sig[:2000], color='blue', linewidth=2)
plt.title("2. Cleaned Signal after Lowpass Filter (5Hz Sine Wave visible)")
plt.ylabel("Amplitude")
plt.grid(True)

# ③ 세 번째: (129, 89) 구조의 소리 지도 (스펙트로그램)
plt.subplot(3, 1, 3)
# pcolormesh는 행렬 데이터를 열지도(Heatmap) 형태로 시각화해 줍니다.
plt.pcolormesh(tt_np, f_np, 10 * np.log10(Sxx_np + 1e-10), shading='gouraud', cmap='viridis')
plt.title(f"3. Spectrogram Image (Shape: {Sxx_np.shape}) - 'Frequency Map'")
plt.ylabel("Frequency (Hz)")
plt.xlabel("Time (seconds)")
plt.ylim(0, 100)  # 우리가 넣은 핵심 소리가 5Hz 주변이므로 하단 영역을 확대해서 봅니다.
plt.colorbar(label="Power (dB)")

plt.tight_layout()
plt.show()

<a id="7"></a>
## 7. special · stats

📖 [`cupyx.scipy.special`](https://docs.cupy.dev/en/stable/reference/scipy_special.html) · [`cupyx.scipy.stats`](https://docs.cupy.dev/en/stable/reference/scipy_stats.html)

### 예제 설명
- expit (시그모이드 함수): 어떤 숫자가 오든 무조건 0과 1 사이의 값으로 변환해 줍니다. AI가 "이 사진이 고양이일 확률은 0.87(87%)이야"라고 확률을 답할 때 마지막 단계에서 쓰는 필수 함수입니다.
- softmax: 여러 개의 선택지(예: 개, 고양이, 호랑이)가 있을 때, 각 선택지의 점수를 "합하면 무조건 1(100%)이 되는 예쁜 확률 분포"로 정돈해 줍니다.
- zscore (표준화): 데이터들의 평균을 0, 표준편차를 1로 맞춰서 데이터의 '체급'을 똑같이 통일해 줍니다. 수능 성적표의 '표준점수'를 만드는 원리이며, AI에게 데이터를 학습시키기 전에 필수로 거치는 전처리 단계입니다.
- trim_mean (절단평균): 심사위원 점수를 매길 때 최고점과 최저점을 몇 %씩 잘라내고 남은 사람들끼리만 평균을 내는 방식입니다. 극단적인 이상치(노이즈)에 평균이 휘청거리는 것을 막아줍니다.


In [ ]:
from cupyx.scipy.special import softmax, logsumexp, expit, erf
from cupyx.scipy.stats import zscore, trim_mean
v = cp.linspace(-6,6,7,dtype=cp.float32)
print('expit:', cp.asnumpy(expit(v)).round(3))
print('softmax 합:', float(softmax(v).sum()))
data = cp.random.random(1_000_000, dtype=cp.float32)
print('zscore mean/std:', float(zscore(data).mean()), float(zscore(data).std()))
print('trim_mean(10%):', float(trim_mean(data, 0.1)))

**연습 — 수치 안정 softmax**: `logsumexp`로 직접 softmax를 구현하고 `special.softmax`와 비교하세요.

In [ ]:
def my_softmax(x):
    # TODO: xp.exp(x - logsumexp(x))
    raise NotImplementedError
# z=cp.random.random(1000,dtype=cp.float32)*20
# allclose(my_softmax(z), softmax(z), rtol=1e-4, atol=1e-6, name='softmax')

<details><summary>💡 해답 보기</summary>

```python
def my_softmax(x):
    return cp.exp(x - logsumexp(x))
z = cp.random.random(1000, dtype=cp.float32)*20
allclose(my_softmax(z), softmax(z), rtol=1e-4, atol=1e-6, name='softmax')
```
</details>

<a id="8"></a>
## 8. 체크포인트

- [ ] `scipy.fft`(DCT)로 압축 데모를 했다
- [ ] `scipy.linalg`(lu_solve·expm)를 GPU에서 실행했다
- [ ] `ndimage`로 연결요소 라벨링·그래디언트를 구했다
- [ ] 희소 CSR 내부구조(data/indices/indptr)를 이해하고 SpMV했다
- [ ] `sparse.linalg`로 2D 푸아송을 직접/반복법으로 풀었다
- [ ] `signal`(butter+filtfilt·spectrogram)·`special`(안정 softmax)을 사용했다

**Day 1 단원 2 완료.** 다음: **`05_memory_profiling`**.